In [13]:
# Install everything required

!pip install pandas numpy scikit-learn xgboost shap matplotlib joblib fastapi uvicorn packaging --quiet

In [14]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV
from packaging import version
import sklearn

In [15]:
# 1 - GENERATE SYNTHETIC HEALTHCARE DATA

np.random.seed(42)

n = 2000

age = np.random.randint(18, 90, n)
gender = np.random.choice(["Male", "Female"], n)
bmi = np.random.normal(26, 5, n)
systolic_bp = np.random.normal(125, 15, n)
diastolic_bp = np.random.normal(80, 10, n)
glucose = np.random.normal(110, 30, n)
cholesterol = np.random.normal(190, 40, n)
smoker = np.random.choice(["yes", "no"], n)

risk = (
    (age > 60).astype(int) +
    (bmi > 30).astype(int) +
    (systolic_bp > 140).astype(int) +
    (glucose > 126).astype(int)
)

recommendation = np.clip(risk, 0, 3)

df = pd.DataFrame({
    "age": age,
    "gender": gender,
    "bmi": bmi,
    "systolic_bp": systolic_bp,
    "diastolic_bp": diastolic_bp,
    "glucose": glucose,
    "cholesterol": cholesterol,
    "smoker": smoker,
    "recommendation": recommendation
})

df.head()

,age,gender,bmi,systolic_bp,diastolic_bp,glucose,cholesterol,smoker,recommendation
0,69,Female,24.685106,116.039529,60.605946,190.415266,189.381166,no,2
1,32,Male,16.821444,117.684890,88.900101,97.268143,150.773484,no,0
2,89,Male,27.650325,131.896709,66.853626,90.040535,248.189981,no,1
3,78,Female,29.099632,122.377695,62.274214,119.607643,265.230446,yes,1
4,38,Female,20.554599,105.062223,76.363867,102.315247,211.875588,no,0


In [16]:
# 2 - BUILD & TRAIN ADVANCED MODEL

X = df.drop("recommendation", axis=1)
y = df["recommendation"]

numeric = X.select_dtypes(include=np.number).columns.tolist()
categorical = X.select_dtypes(exclude=np.number).columns.tolist()

# Version-safe OHE
if version.parse(sklearn.__version__) >= version.parse("1.2"):
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
else:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", ohe)
    ]), categorical)
])

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    eval_metric="mlogloss",
    random_state=42
)

calibrated = CalibratedClassifierCV(xgb, method="isotonic", cv=3)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", calibrated)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=42
)

pipeline.fit(X_train, y_train)

print("Model training complete.")

Model training complete.


In [17]:
# 3 - CLINICAL RULE ENGINE

def clinical_rule_engine(patient):
    rules = []

    if patient.get("glucose", 0) > 126:
        rules.append("Recommend HbA1c test.")

    if patient.get("systolic_bp", 0) > 140:
        rules.append("Monitor blood pressure daily.")

    if patient.get("bmi", 0) > 30:
        rules.append("Weight management advised.")

    if patient.get("smoker") == "yes":
        rules.append("Smoking cessation counseling.")

    return rules

In [18]:
# 4 - DRIFT DETECTION + LOGGING

training_means = X_train[numeric].mean().to_dict()
prediction_logs = []

def detect_drift(new_sample):
    drift = {}
    for k, v in training_means.items():
        if k in new_sample:
            if abs(new_sample[k] - v) > abs(v) * 0.3:
                drift[k] = "Possible drift detected"
    return drift

def log_prediction(record):
    prediction_logs.append(record)

In [19]:
# 5 - ADVANCED PREDICTION FUNCTION (WITH CONFIDENCE GATING)

CONFIDENCE_THRESHOLD = 0.60

RECOMMENDATION_MAP = {
    0: "Routine monitoring.",
    1: "Lifestyle modification advised.",
    2: "Specialist consultation recommended.",
    3: "Urgent clinician review required."
}

def predict_patient(patient_dict):

    df_input = pd.DataFrame([patient_dict])

    probs = pipeline.predict_proba(df_input)[0]
    pred = int(np.argmax(probs))
    confidence = float(np.max(probs))

    if confidence < CONFIDENCE_THRESHOLD:
        final_text = "Low confidence — refer to clinician."
    else:
        final_text = RECOMMENDATION_MAP[pred]

    rules = clinical_rule_engine(patient_dict)
    drift_flags = detect_drift(patient_dict)

    record = {
        "input": patient_dict,
        "prediction": pred,
        "confidence": confidence,
        "rules": rules,
        "drift": drift_flags
    }

    log_prediction(record)

    return record

In [20]:
# 6 - TEST THE SYSTEM

example_patient = {
    "age": 67,
    "gender": "Male",
    "bmi": 32,
    "systolic_bp": 155,
    "diastolic_bp": 95,
    "glucose": 160,
    "cholesterol": 250,
    "smoker": "yes"
}

result = predict_patient(example_patient)

result

{'input': {'age': 67,
  'gender': 'Male',
  'bmi': 32,
  'systolic_bp': 155,
  'diastolic_bp': 95,
  'glucose': 160,
  'cholesterol': 250,
  'smoker': 'yes'},
 'prediction': 3,
 'confidence': 1.0,
 'rules': ['Recommend HbA1c test.',
  'Monitor blood pressure daily.',
  'Weight management advised.',
  'Smoking cessation counseling.'],
 'drift': {'glucose': 'Possible drift detected',
  'cholesterol': 'Possible drift detected'}}